In [ ]:
# ============================================================
# Requirement ID : Q1
# Requirement    : Combine Raw HDB Resale Datasets into Master Dataset
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 18-Aug-2026
# ============================================================

# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import sys
import logging

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

from pyspark.sql.functions import (col,trim,upper,when,lit,date_format,to_date,split,floor,expr,row_number,lpad,
    regexp_replace,substring, concat, sha2, countDistinct, min,max, mean,stddev, sum as spark_sum, approx_count_distinct)

# ============================================================
# LOGGING CONFIGURATION
# ============================================================

# Main project folder
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Logs folder under main project folder
LOG_DIR = os.path.join(BASE_DIR, "logs")

# Create logs folder if it does not exist
os.makedirs(LOG_DIR, exist_ok=True)

# Current date
LOG_DATE = datetime.now().strftime("%Y-%m-%d")

# Daily log file
LOG_FILE = os.path.join(
    LOG_DIR,
    f"HDB_Resale_ETL_{LOG_DATE}.log"
)

# Configure logging
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    filemode="a"
)

logger = logging.getLogger(__name__)

# Pipeline start
logger.info("============================================================")
logger.info("HDB RESALE ETL PIPELINE STARTED")
logger.info("============================================================")

print("Log file:", LOG_FILE)


try:

    logger.info("Q1 Dataset Merging process started.")

    # ============================================================
    # PYTHON CONFIGURATION FOR SPARK - WINDOWS
    # ============================================================
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

    # ============================================================
    # INITIALIZE SPARK SESSION
    # ============================================================
    
    # Stop any existing Spark session to avoid "Only one SparkContext should be running" errors
    try:
        spark.stop()
    except:
        pass
    
    # Create a new SparkSession for the ETL pipeline
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("HDB_Resale_ETL_Pipeline") \
        .config("spark.hadoop.hadoop.native.lib", "false") \
        .config("spark.hadoop.io.native.lib.available", "false") \
        .getOrCreate()
    
    print("Spark Session Initialized")
    print("Spark Version:", spark.version)
    
    
    # ============================================================
    # LOAD ENVIRONMENT VARIABLES
    # ============================================================
    
    # Load key-value pairs from the .env file into environment variables
    load_dotenv("hdb_resale.env")
    
    # Retrieve dataset paths from environment variables
    Rawdata_2000_to_Feb2012 = os.getenv("Rawdata_2000_to_Feb2012")   # Path for resale data covering 2000–Feb 2012
    Rawdata_Mar2012_to_Dec2014 = os.getenv("Rawdata_Mar2012_to_Dec2014")   # Path for resale data covering Mar 2012–Dec 2014
    Rawdata_Jan2015_to_Dec2016 = os.getenv("Rawdata_Jan2015_to_Dec2016")   # Path for resale data covering Jan 2015–Dec 2016
    
    # Retrieve output path for the final master dataset
    Master_Dataset_Path = os.getenv("HDB_Master_Raw")   # Destination path for merged Master_Dataset.csv

    # DataFrame for profiling and processing
    HDB_Dataset_Cleaned  = os.getenv("HDB_Cleaned")    
    HDB_Dataset_Failed  = os.getenv("HDB_Failed")    

    # HDB Lease Recalculation
    HDB_Dataset_Cleaned_With_Lease = os.getenv("HDB_Dataset_Cleaned_With_Lease")

    #HDB Resale Price Anomaly and Normal Detection
    HDB_ResalePrice_Anomaly_Detection = os.getenv("HDB_ResalePrice_Anomaly_Detection")
    
    # Addtional Data Cleaning & Validation Rules
    HDB_Addtional_Data_Cleaning = os.getenv("HDB_Addtional_Data_Cleaning")

    # Data Transformation and Hashed
    HDB_Transformed_Hashed = os.getenv("HDB_Transformed_Hashed")

    # ============================================================
    # VALIDATE ENVIRONMENT VARIABLES
    # ============================================================
    
    # Check if the path for 2000–Feb 2012 dataset is missing
    # Raise an error immediately if not found
    if not Rawdata_2000_to_Feb2012:
        raise ValueError("Missing environment variable: Rawdata_2000_to_Feb2012")
    
    # Check if the path for Mar 2012–Dec 2014 dataset is missing
    # Prevents pipeline from running with incomplete inputs
    if not Rawdata_Mar2012_to_Dec2014:
        raise ValueError("Missing environment variable: Rawdata_Mar2012_to_Dec2014")
    
    # Check if the path for Jan 2015–Dec 2016 dataset is missing
    # Ensures all three raw datasets are available before proceeding
    if not Rawdata_Jan2015_to_Dec2016:
        raise ValueError("Missing environment variable: Rawdata_Jan2015_to_Dec2016")
    
    
    # ============================================================
    # LOAD RAW DATASETS
    # ============================================================
    
    # Load raw resale dataset covering 2000–Feb 2012
    df_jan_feb_2012 = spark.read.option("header", True).option("inferSchema", True).csv(Rawdata_2000_to_Feb2012)
    
    # Load raw resale dataset covering Mar 2012–Dec 2014
    df_mar2012_dec2014 = spark.read.option("header", True).option("inferSchema", True).csv(Rawdata_Mar2012_to_Dec2014)
    
    # Load raw resale dataset covering Jan 2015–Dec 2016
    df_jan2015_dec2016 = spark.read.option("header", True).option("inferSchema", True).csv(Rawdata_Jan2015_to_Dec2016)
    
    print("\nRaw datasets loaded successfully.")
    
    
    # ============================================================
    # FIX MONTH FORMAT TO YYYY-MM
    # ============================================================
    
    def fix_month_format(df):
        return df.withColumn("month",date_format(to_date(col("month")),"yyyy-MM"))
    
    # Standardize month format for Jan–Feb 2012 dataset
    df_jan_feb_2012 = fix_month_format(df_jan_feb_2012)
    
    # Standardize month format for Mar 2012–Dec 2014 dataset
    df_mar2012_dec2014 = fix_month_format(df_mar2012_dec2014)
    
    # Standardize month format for Jan 2015–Dec 2016 dataset
    df_jan2015_dec2016 = fix_month_format(df_jan2015_dec2016)
    
    # ============================================================
    # FILTER JANUARY AND FEBRUARY 2012
    # ============================================================
    
    # Keep only records from January and February 2012
    # - col("month") == "2012-01" → selects January 2012
    # - col("month") == "2012-02" → selects February 2012
    # - The filter ensures that only these two months remain in df_jan_feb_2012
    df_jan_feb_2012 = df_jan_feb_2012.filter((col("month") == "2012-01") | (col("month") == "2012-02"))
    
    # ============================================================
    # MERGE ALL DATASETS
    # ============================================================
    
    # Combine the three cleaned datasets into one unified DataFrame
    # - unionByName ensures columns are matched by name, not position
    # - allowMissingColumns=True allows schema differences (missing columns) to be handled gracefully
    master_df = (
        df_jan_feb_2012
        .unionByName(df_mar2012_dec2014,allowMissingColumns=True)
        .unionByName(df_jan2015_dec2016,allowMissingColumns=True
        )
    )
    
    # ============================================================
    # CREATE FINAL MASTER DATASET
    # ============================================================
    
    # Select only the required columns from the combined DataFrame
    # - month: resale transaction month (standardized to YYYY-MM)
    # - town: HDB town/estate
    # - flat_type: type of flat (e.g., 3-room, 4-room)
    # - block: block number
    # - street_name: street where the flat is located
    # - storey_range: floor range (e.g., 10 to 12)
    # - floor_area_sqm: size of the flat in square meters
    # - flat_model: model type of the flat
    # - lease_commence_date: year the lease started
    # - remaining_lease: number of years left on the lease
    # - resale_price: transaction price of the resale flat
    master_df = master_df.select(
        "month","town","flat_type","block","street_name","storey_range","floor_area_sqm","flat_model",
        "lease_commence_date","remaining_lease","resale_price"
    )
    
    # ============================================================
    # VALIDATION
    # ============================================================
    
    print("\n============================================================")
    print("VALIDATION CHECKS")
    print("============================================================")
    
    # Count total number of records in the master dataset
    total_records = master_df.count()
    print(f"Total Records: {total_records:,}")
    
    # Check for null values in each column
    # - col(c).isNull().cast("int") → returns 1 if value is null, else 0
    # - alias(c) → names the column for clarity
    # - groupBy().sum() → aggregates null counts across all rows
    print("\nNull Counts per Column:")
    null_counts = master_df.select(
        [
            col(c).isNull().cast("int").alias(c)
            for c in master_df.columns
        ]
    ).groupBy().sum()
    
    # Display null counts for each column
    null_counts.show()
    
    # Show a sample of 5 records from the merged dataset
    print("\n===== MERGED DATA SAMPLE =====")
    master_df.show(5, truncate=False)
    
    # ============================================================
    # SAVE FINAL CSV TO WINDOWS
    #
    # Spark performs the ETL.
    # Pandas is used ONLY for the final local Windows CSV export
    # because Hadoop NativeIO is failing on this Windows setup.
    # ============================================================
    
    print("\n============================================================")
    print("SAVING MASTER DATASET")
    print("============================================================")
    
    print("Converting Spark DataFrame to Pandas...")
    master_pd = master_df.toPandas()
    print("Conversion completed.")
   
    # Extract folder path
    output_folder = os.path.dirname(Master_Dataset_Path)
    
    # Ensure folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Remove existing file if present
    if os.path.isfile(Master_Dataset_Path):
        try:
            os.remove(Master_Dataset_Path)
            print("Existing Dataset removed.")
        except PermissionError:
            raise PermissionError(
                "\n Cannot overwrite Dataset.\n\n"
                "Please close the file in Excel/File Explorer "
                "and run the notebook again."
            )
    
    # Save cleaned dataset
    master_pd.to_csv(Master_Dataset_Path, index=False, encoding="utf-8-sig")
    
    # Final validation
    if not os.path.exists(Master_Dataset_Path):
        raise FileNotFoundError("Dataset was not created.")
    
    file_size = os.path.getsize(Master_Dataset_Path)
    print(f"Dataset created successfully. Size: {file_size} bytes")
    
    # ============================================================
    # FINAL SAMPLE
    # ============================================================
    
    print("\n===== FINAL MASTER DATASET SAMPLE =====")   
    master_df.show(5, truncate=False)
    
    logger.info(f"Cleaned records: {master_df.count()}")
    logger.info(f"Q1 Dataset merging completed successfully.\n")

    print("\n============================================================")
    print("Q1 DATASET MERGING COMPLETED")
    print("============================================================")

except Exception as e:
    logger.exception("Q1 Dataset merging failed")
    print("Q1 Dataset merging failed")
    print("Error:", e)
    raise

In [ ]:
# ============================================================
# Requirement ID : Q2
# Requirement    : Perform Data Profiling on the Master Dataset
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 18-Aug-2026
# ============================================================

# ============================================================
# Q2: DATA PROFILING
# ============================================================
#
# Objective:
# Profile the master dataset to understand its structure, quality,
# completeness, uniqueness, statistical characteristics, and potential
# data anomalies before applying data validation and transformation rules.
#
# The profiling process includes:
# 1. Basic dataset overview
# 2. Missing value analysis
# 3. Duplicate record analysis
# 4. Data type profiling
# 5. Unique value profiling
# 6. Numeric statistical profiling
# 7. Resale price outlier detection
# 8. Categorical value distribution analysis
# 9. Numeric consistency checks
# 10. Date quality checks
# 11. Correlation analysis
# ============================================================

try:
    logger.info("Q2 Data profiling process started.")
    # ============================================================
    # 1. BASIC DATASET OVERVIEW
    # ============================================================
    
    # Review the overall structure of the master dataset, including
    # the total number of records, schema, and a sample of the data.
    print("\n--- Basic Dataset Overview ---")

    total_records = master_df.count()
    print("Total Records:", total_records)

    print("\n--- Dataset Schema ---")
    master_df.printSchema()

    print("\n--- Sample Data ---")
    master_df.show(5, truncate=False)


    # ============================================================
    # 2. MISSING VALUE PROFILING
    # ============================================================

    # Calculate the number and percentage of null values for each column
    # to assess data completeness.
    print("\n--- Missing Value Profile ---")

    missing_profile = master_df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in master_df.columns
    ])

    missing_profile.show(truncate=False)

    missing_percent = master_df.select([
        (spark_sum(col(c).isNull().cast("int")) / total_records * 100).alias(c)
        for c in master_df.columns
    ])

    print("\n--- Missing Value Percentages ---")
    missing_percent.show(truncate=False)


    # ============================================================
    # 3. DUPLICATE RECORD PROFILING
    # ============================================================

    # Compare the total number of records with the number of distinct
    # records to identify exact duplicate rows in the master dataset.
    print("\n--- Duplicate Record Profile ---")

    total_rows = master_df.count()
    unique_rows = master_df.dropDuplicates().count()
    duplicate_rows = total_rows - unique_rows

    duplicate_profile = spark.createDataFrame(
        [("Duplicate Rows", duplicate_rows)],
        ["Metric", "Value"]
    )

    duplicate_profile.show(truncate=False)


    # ============================================================
    # 4. DATA TYPE PROFILING
    # ============================================================

    # Display the data type of each column to verify that attributes
    # have been loaded with the expected schema.
    print("\n--- Data Type Profile ---")

    for field in master_df.schema.fields:
        print(f"{field.name}: {field.dataType}")


    # ============================================================
    # 5. UNIQUE VALUE PROFILING
    # ============================================================

    # Estimate the number of distinct values in each column to understand
    # cardinality and identify highly repetitive or highly unique attributes.
    print("\n--- Unique Value Profile ---")

    unique_counts = master_df.agg(
        *[
            approx_count_distinct(col(c)).alias(c)
            for c in master_df.columns
        ]
    )

    unique_counts.show(truncate=False)


    # ============================================================
    # 6. NUMERIC STATISTICAL PROFILING
    # ============================================================

    # Generate descriptive statistics for numeric columns, including
    # count, mean, standard deviation, minimum, and maximum values.
    print("\n--- Numeric Statistical Profile ---")

    master_df.describe().show()


    # ============================================================
    # 7. RESALE PRICE OUTLIER PROFILING
    # ============================================================

    # Use the Interquartile Range (IQR) method to identify potentially
    # unusual resale prices for further investigation.
    #
    # Lower Bound = Q1 - 1.5 * IQR
    # Upper Bound = Q3 + 1.5 * IQR
    print("\n--- Resale Price Outlier Profile ---")

    stats = master_df.select(
        mean(col("resale_price")).alias("mean"),
        stddev(col("resale_price")).alias("stddev"),
        min(col("resale_price")).alias("min"),
        max(col("resale_price")).alias("max")
    ).collect()[0]

    q1 = master_df.approxQuantile("resale_price", [0.25], 0.01)[0]
    q3 = master_df.approxQuantile("resale_price", [0.75], 0.01)[0]
    iqr = q3 - q1

    outliers = master_df.filter(
        (col("resale_price") < (q1 - 1.5 * iqr)) |
        (col("resale_price") > (q3 + 1.5 * iqr))
    )

    print("Outlier Count:", outliers.count())
    outliers.show(5, truncate=False)


    # ============================================================
    # 8. CATEGORICAL VALUE PROFILING
    # ============================================================

    # Analyse the frequency distribution of key categorical attributes
    # to understand common values and identify potentially unusual values.
    print("\n--- Categorical Value Profile ---")

    print("Flat Type Distribution:")
    master_df.groupBy("flat_type").count().orderBy(
        col("count").desc()
    ).show()

    print("\nStorey Range Distribution:")
    master_df.groupBy("storey_range").count().orderBy(
        col("count").desc()
    ).show(10)


    # ============================================================
    # 9. NUMERIC CONSISTENCY PROFILING
    # ============================================================

    # Review the minimum and maximum values of key numeric attributes
    # to identify potentially invalid or unexpected ranges.
    print("\n--- Numeric Consistency Profile ---")

    print("Resale Price Range:")
    master_df.select(
        min("resale_price").alias("min_resale_price"),
        max("resale_price").alias("max_resale_price")
    ).show()

    print("Floor Area Range:")
    master_df.select(
        min("floor_area_sqm").alias("min_floor_area_sqm"),
        max("floor_area_sqm").alias("max_floor_area_sqm")
    ).show()


    # ============================================================
    # 10. DATE PROFILING
    # ============================================================

    # Check for null or unparseable transaction month values.
    # The expected transaction period for this assignment is from
    # January 2012 to December 2016.
    print("\n--- Date Profile ---")

    invalid_dates = master_df.filter(col("month").isNull())

    print("Invalid or Null Date Rows:", invalid_dates.count())


    # ============================================================
    # 11. CORRELATION ANALYSIS
    # ============================================================

    # Measure the linear relationship between floor area and resale price.
    # This provides an addtional statistical view of the dataset and may
    # help identify unexpected data patterns.
    print("\n--- Correlation Analysis ---")

    corr_value = master_df.stat.corr(
        "floor_area_sqm",
        "resale_price"
    )

    print(
        "Correlation between floor_area_sqm and resale_price:",
        corr_value
    )


    # ============================================================
    # 12. PROFILING SUMMARY
    # ============================================================

    # Summarize key profiling results for use in subsequent validation
    # and data cleaning steps.
    print("\n============================================================")
    print("Q2: DATA PROFILING SUMMARY")
    print("============================================================")
    print("Total Rows:", total_records)
    print("Duplicate Rows:", duplicate_rows)
    print("Potential Resale Price Outliers:", outliers.count())
    print("============================================================")    
    
    logger.info(f"Total Rows: {total_records}")
    logger.info(f"Duplicate Rows: {duplicate_rows}")
    logger.info(f"Potential Resale Price Outliers: {outliers.count()}")
    logger.info(f"Q2 Data profiling completed successfully.\n")

    print("\n============================================================")
    print("Q2 DATA PROFILING COMPLETED")
    print("============================================================")
    
except Exception as e:
    logger.exception("Q2 Data profiling failed")
    print("Q2 Data profiling failed")
    print("Error:", e)
    raise


In [ ]:
#pip install ydata-profiling
#python.exe -m pip install

In [ ]:
# ============================================================
# Requirement ID : Q2
# Requirement    : Perform Data Profiling on the Master Dataset
# Input          : master_df
# Output         : Profiling summary + HTML profiling report
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 19-Aug-2026
# ============================================================

from ydata_profiling import ProfileReport

try:

    print("\n============================================================")
    print("Q2: DATA PROFILING")
    print("============================================================")

    logger.info("Q2 YData Profiling started")

    # ------------------------------------------------------------
    # 1. Basic Dataset Information
    # ------------------------------------------------------------

    total_records = master_df.count()
    total_columns = len(master_df.columns)

    print("Total Records :", total_records)
    print("Total Columns :", total_columns)

    logger.info(f"Q2 Total records: {total_records}")
    logger.info(f"Q2 Total columns: {total_columns}")


    # ------------------------------------------------------------
    # 2. Dataset Schema
    # ------------------------------------------------------------

    print("\n===== DATA TYPES =====")
    master_df.printSchema()


    # ------------------------------------------------------------
    # 3. Sample Records
    # ------------------------------------------------------------

    print("\n===== SAMPLE DATA =====")
    master_df.show(5, truncate=False)


    # ------------------------------------------------------------
    # 4. Null Value Profiling
    # ------------------------------------------------------------

    print("\n===== NULL COUNTS =====")

    null_counts_df = master_df.select([
        F.sum(
            F.when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in master_df.columns
    ])

    null_counts_df.show(truncate=False)


    # ------------------------------------------------------------
    # 5. Duplicate Record Count
    # ------------------------------------------------------------

    duplicate_count = (total_records - master_df.dropDuplicates().count())
    print("Duplicate Records :", duplicate_count)
    logger.info(f"Q2 Duplicate records: {duplicate_count}")


    # ------------------------------------------------------------
    # 6. YData Profiling Report
    # ------------------------------------------------------------

    print("\n============================================================")
    print("GENERATING YDATA PROFILING REPORT")
    print("============================================================")

    # Create reports folder under main project folder
    REPORT_DIR = os.path.join(BASE_DIR, "reports")
    os.makedirs(REPORT_DIR, exist_ok=True)

    # Convert Spark DataFrame to Pandas only for profiling
    profiling_pd = master_df.toPandas()

    # Generate profiling report
    profile = ProfileReport(profiling_pd, title="HDB Resale Data Profiling Report", explorative=True)

    # Report output path
    PROFILE_FILE = os.path.join(REPORT_DIR,"HDB_Data_Profiling_Report.html")

    # Save report
    profile.to_file(PROFILE_FILE)

    print("YData Profiling report generated successfully.")
    print("Report:", PROFILE_FILE)

    logger.info(f"YData Profiling report generated: {PROFILE_FILE}")


    # ------------------------------------------------------------
    # 7. Q2 Completion
    # ------------------------------------------------------------

    logger.info("Q2 YData Profiling completed successfully.\n")

    print("\n============================================================")
    print("Q2 YDATA PROFILING COMPLETED SUCCESSFULLY")
    print("============================================================")

except Exception as e:
    logger.exception("Q2 YData profiling failed")
    print("Q2 YData profiling failed")
    print("Error:", e)
    raise


In [ ]:
# ============================================================
# Requirement ID : Q3
# Requirement    : Perform Data Validation on the Master Dataset
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 19-Aug-2026
# ============================================================


try:
    logger.info("Q3 Data validation process started.")
    
    print("\n============================================================")
    print("Q3: DATA VALIDATION RULES")
    print("============================================================")


    # ------------------------------------------------------------
    # 1. NORMALISE COLUMNS
    # ------------------------------------------------------------
    
    master_df = (
        master_df
        .withColumn("month_clean", trim(col("month").cast("string")))
        .withColumn("town_clean", upper(trim(col("town"))))
        .withColumn("flat_type_clean", upper(trim(col("flat_type"))))
        .withColumn("flat_model_clean", upper(trim(col("flat_model"))))
        .withColumn("storey_range_clean", upper(trim(col("storey_range"))))
    )
    

    # ------------------------------------------------------------
    # 2. BASIC DATA PROFILING
    # ------------------------------------------------------------

    total_records = master_df.count()
    #print("Total records:", total_records)

    # ------------------------------------------------------------
    # 3. DATE FIELD VALIDATION (MONTH)
    # ------------------------------------------------------------     
    # Rules:
    #   - month must not be null
    #   - month must follow YYYY-MM format
    #   - date must be between Jan 2012 and Dec 2016
    # ------------------------------------------------------------

    validated_df = (master_df
                    .withColumn("month_clean",trim(col("month").cast("string")))
                    .withColumn("month_date",F.to_date(col("month_clean"),"yyyy-MM")
        )
    )


    validated_df = validated_df.withColumn(
        "is_valid_date",
        (
            col("month_clean").isNotNull() &
            (col("month_clean") != "")&
            col("month_date").isNotNull()&
            (col("month_date") >= F.to_date(lit("2012-01-01")))&
            (col("month_date") <= F.to_date(lit("2016-12-01")))
        )
    )
    
    #validated_df.show(5, truncate=False)

    # ------------------------------------------------------------
    # 4. TOWN VALIDATION AND FREQUENCY PROFILING
    # ------------------------------------------------------------
    # Null / blank values are invalid.
    # Frequency is profiled separately if required.
    # ------------------------------------------------------------

    validated_df = validated_df.withColumn("is_valid_town",
        (col("town_clean").isNotNull()
         &(col("town_clean") != "")
        )
    )
    #validated_df.show(5, truncate=False)

    # ------------------------------------------------------------
    # 5. FLAT TYPE VALIDATION AND FREQUENCY PROFILING
    # ------------------------------------------------------------

    validated_df= validated_df.withColumn("is_valid_flat_type",(col("flat_type_clean").isNotNull()
                                                &(col("flat_type_clean") != "")
                                               )
                         )
                        
    #validated_df.show(5, truncate=False)                

    # ------------------------------------------------------------
    # 6. FLAT MODEL VALIDATION
    # ------------------------------------------------------------

    validated_df = validated_df.withColumn("is_valid_flat_model",(col("flat_model_clean").isNotNull()
                                                                  &(col("flat_model_clean") != "")))
    
    #validated_df.show(5, truncate=False)
    
    # ------------------------------------------------------------
    # 7. STOREY RANGE VALIDATION
    # ------------------------------------------------------------
    #
    # Expected format:
    #
    #   01 TO 03
    #   04 TO 06
    #   07 TO 09
    #
    # Validation:
    #   - Correct format
    #   - Lower floor > 0
    #   - Upper floor >= lower floor
    # ------------------------------------------------------------

    validated_df = validated_df.withColumn("storey_range_clean",upper(trim(col("storey_range"))))

    validated_df = (validated_df
                    .withColumn("storey_lower",F.regexp_extract(col("storey_range_clean"),r"^([0-9]{2}) TO ([0-9]{2})$",1).cast("int"))
                    .withColumn("storey_upper",F.regexp_extract(col("storey_range_clean"),r"^([0-9]{2}) TO ([0-9]{2})$",2).cast("int")
                               )
                   )
    
    validated_df = validated_df.withColumn("is_valid_storey",(col("storey_range_clean").isNotNull()
                                                              &col("storey_range_clean").rlike(r"^[0-9]{2} TO [0-9]{2}$")
                                                              &(col("storey_lower") > 0)
                                                              &(col("storey_upper") >= col("storey_lower"))
                                                             )
                                          )
    #validated_df.show(5, truncate=False)       

    # ------------------------------------------------------------
    # 8. OVERALL VALIDATION FLAG
    # ------------------------------------------------------------
   

    validated_df = validated_df.withColumn("is_valid_record",(col("is_valid_date")
                                          &col("is_valid_town")
                                          &col("is_valid_flat_type")
                                          &col("is_valid_flat_model")
                                          &col("is_valid_storey")
                                         )
                      )      
    
    
    # ------------------------------------------------------------
    # 9. FAILURE REASON
    # ------------------------------------------------------------

    validated_df = validated_df.withColumn("failed_rule",F.concat_ws("; ",
                                                                        when(~col("is_valid_date"),lit("Invalid date")),
                                                                        when(~col("is_valid_town"),lit("Invalid town")),
                                                                        when(~col("is_valid_flat_type"),lit("Invalid flat type")),
                                                                        when(~col("is_valid_flat_model"),lit("Invalid flat model")),
                                                                        when(~col("is_valid_storey"),lit("Invalid storey range"))
                                                                       )
                                          )
    
    #validated_df.show(5, truncate=False)

    # ------------------------------------------------------------
    # 10. CLEANED DATASET
    # ------------------------------------------------------------
    cleaned_df = validated_df.filter(col("is_valid_record"))
    #cleaned_df.show(5, truncate=False)

    # ------------------------------------------------------------
    # 11. FAILED DATASET
    # ------------------------------------------------------------
    failed_df = validated_df.filter(~col("is_valid_record"))
    #failed_df.show(5, truncate=False)

    # ------------------------------------------------------------
    # 12. VALIDATION SUMMARY
    # ------------------------------------------------------------

    print("\n============================================================")
    print("Q3 VALIDATION SUMMARY")
    print("============================================================")

    print("Total records:", validated_df.count())
    print("Invalid date:",validated_df.filter(~col("is_valid_date")).count())
    print("Invalid town:",validated_df.filter(~col("is_valid_town")).count())
    print("Invalid flat type:",validated_df.filter(~col("is_valid_flat_type")).count())
    print("Invalid flat model:",validated_df.filter(~col("is_valid_flat_model")).count())
    print("Invalid storey:",validated_df.filter(~col("is_valid_storey")).count())
    print("Cleaned records:",cleaned_df.count())
    print("Failed records:",failed_df.count())

    # ------------------------------------------------------------
    # 13. SHOW FAILED RECORDS
    # ------------------------------------------------------------

    print("\n============================================================")
    print("FAILED RECORDS")
    print("============================================================")

    failed_df.select("month","town","flat_type","flat_model","storey_range","failed_rule").show(20,truncate=False)

    print("\n============================================================")
    print("Saving Final Dataset")
    print("============================================================")
      
    cleaned_df.toPandas().to_csv(HDB_Dataset_Cleaned, index=False, encoding="utf-8-sig")
    failed_df.toPandas().to_csv(HDB_Dataset_Failed, index=False, encoding="utf-8-sig")
    
    print("\nFiles saved successfully:")
    print("Valid dataset ->", HDB_Dataset_Cleaned)
    print("Invalid dataset ->", HDB_Dataset_Failed)
    
    logger.info(f"Valid Data Rows: {cleaned_df.count()}")
    logger.info(f"Failed Data Rows: {failed_df.count()}")
    logger.info(f"Q3 Data validation completed successfully.\n")

    print("\n============================================================")
    print("Q3 DATA VALIDATION COMPLETED")
    print("============================================================")

except Exception as e:
    logger.exception("Q3 Data validation failed")
    print("Q3 Data validation failed")
    print("Error:", e)
    raise


In [ ]:
# ============================================================
# Requirement ID : Q4
# Requirement    : Recompute Remaining Lease
# Input          : cleaned_df from Q3
# Output         : lease_dataset_df
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 19-Aug-2026
# ============================================================

try:

    logger.info("Q4 Recompute remaining lease process started.")
    
    print("\n============================================================")
    print("Q4: RECOMPUTE REMAINING LEASE")
    print("============================================================")


    # ------------------------------------------------------------
    # 1. USE VALID DATASET FROM Q3
    # ------------------------------------------------------------
    # Q3 cleaned_df contains records that passed validation.
    # ------------------------------------------------------------

    q4_input_df = cleaned_df
    print("Q4 input dataset row count:",q4_input_df.count())
    
    # ------------------------------------------------------------
    # 2. DEFINE CURRENT DATE
    # ------------------------------------------------------------
    today = datetime.today()
    current_year = today.year
    current_month = today.month

    print("Current year :",current_year)
    print("Current month:",current_month)


    # ------------------------------------------------------------
    # 3. COMPUTE LEASE EXPIRY YEAR
    # ------------------------------------------------------------
    #
    # Assumption:
    # HDB lease period = 99 years.
    #
    # Lease expiry year =
    # lease_commence_date + 99 years
    # ------------------------------------------------------------

    lease_dataset_df = q4_input_df.withColumn("lease_expiry_year",col("lease_commence_date") + 99)


    # ------------------------------------------------------------
    # 4. COMPUTE REMAINING MONTHS
    # ------------------------------------------------------------
    #
    # Remaining months =
    # (expiry year - current year) * 12
    # + (12 - current month)
    # ------------------------------------------------------------
    lease_dataset_df = lease_dataset_df.withColumn("remaining_months",((col("lease_expiry_year") - current_year) * 12 + (12 - current_month)))

    # ------------------------------------------------------------
    # 5. SPLIT REMAINING MONTHS INTO YEARS AND MONTHS
    # ------------------------------------------------------------

    lease_dataset_df = (lease_dataset_df
                        .withColumn("remaining_lease",F.floor(col("remaining_months") / 12))
                        .withColumn("remaining_months_final",F.pmod(col("remaining_months"),12))
                       )


    # ------------------------------------------------------------
    # 6. CREATE FORMATTED REMAINING LEASE
    # ------------------------------------------------------------

    lease_dataset_df = lease_dataset_df.withColumn("recomputed_remaining_lease",
                                                   F.concat(col("remaining_lease").cast("string"),
                                                            F.lit(" years "),
                                                            col("remaining_months_final").cast("string"),
                                                            F.lit(" months")
                                                           )
                                                  )

    # ------------------------------------------------------------
    # 7. VALIDATION SAMPLE
    # ------------------------------------------------------------

    print("\n============================================================")
    print("RECOMPUTED REMAINING LEASE SAMPLE")
    print("============================================================")

    lease_dataset_df.select("lease_commence_date","lease_expiry_year", "remaining_months", 
                            "remaining_lease","remaining_months_final","recomputed_remaining_lease").show(10,truncate=False)


    # ------------------------------------------------------------
    # 8. FINAL ROW COUNT
    # ------------------------------------------------------------

    print("\n============================================================")
    print("Q4 SUMMARY")
    print("============================================================")

    print("Input records :",q4_input_df.count())
    print("Output records:",lease_dataset_df.count())


    # ------------------------------------------------------------
    # 9. SAVE DATASET
    # ------------------------------------------------------------

    print("\n============================================================")
    print("SAVING Q4 DATASET")
    print("============================================================")

    lease_dataset_df.toPandas().to_csv(HDB_Dataset_Cleaned_With_Lease,index=False,encoding="utf-8-sig")
    print("\nFile saved successfully:")
    print("Recomputed lease dataset ->",HDB_Dataset_Cleaned_With_Lease)
     
    
    lease_dataset_df.toPandas().to_csv(HDB_Dataset_Cleaned, index=False, encoding="utf-8-sig")
    print("\nFiles saved successfully:")
    print("Valid dataset ->", HDB_Dataset_Cleaned)
   

    # ------------------------------------------------------------
    # 10. FILE SIZE
    # ------------------------------------------------------------

    lease_file_size = os.path.getsize(HDB_Dataset_Cleaned_With_Lease)
    print(f"Dataset created successfully. "f"Size: {lease_file_size} bytes")


    # ------------------------------------------------------------
    # 11. FINAL PREVIEW
    # ------------------------------------------------------------

    print("\n============================================================")
    print("FINAL Q4 DATASET SAMPLE")
    print("============================================================")
    lease_dataset_df.show(5,truncate=False)


    logger.info(f"Valid Data Rows: {lease_dataset_df.count()}")    
    logger.info(f"Q4 Recompute remaining lease completed successfully.\n")
    
    print("\n============================================================")
    print("Q4 RECOMPUTE REMAINING LEASE COMPLETED SUCCESSFULLY")
    print("============================================================")

except Exception as e:
    logger.exception("Q4 Recompute remaining lease failed")
    print("Q4 Recompute remaining lease failed")
    print("Error:", e)
    raise


In [ ]:
# ============================================================
# Requirement ID : Q5
# Requirement    : Duplicate Handling
# Input          : cleaned_df from Q4
# Output         : valid_dataset_df + final_failed_df
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 20-Aug-2026
# ============================================================



try:

    logger.info("Q5 Duplicate data handling process started.")
    print("\n============================================================")
    print("Q5: DUPLICATE HANDLING")
    print("============================================================")


    # ------------------------------------------------------------
    # 1. USE CLEANED DATASET FROM Q3
    # ------------------------------------------------------------
    #
    # cleaned_df contains only records that passed Q3 validation.
    # Do NOT use master_df here.
    # ------------------------------------------------------------

    q5_input_df = lease_dataset_df
    print("Q5 input records:",q5_input_df.count())


    # ------------------------------------------------------------
    # 2. DEFINE COMPOSITE KEY
    # ------------------------------------------------------------
    #
    # All columns except resale_price are used to identify
    # potential duplicate business records.
    # ------------------------------------------------------------

    business_keys = [
        c for c in q5_input_df.columns
        if c not in [
            "resale_price",
            "is_valid_date",
            "is_valid_town",
            "is_valid_flat_type",
            "is_valid_flat_model",
            "is_valid_storey",
            "is_valid_record",
            "failed_rule"
        ]
    ]

    print("\nComposite key columns:")
    print(business_keys)


    # ------------------------------------------------------------
    # 3. CREATE WINDOW
    # ------------------------------------------------------------
    #
    # For records having the same business key:
    #
    # highest resale_price = retained
    # lower resale_price    = duplicate / failed
    # ------------------------------------------------------------

    w = (Window.partitionBy(*[col(c) for c in business_keys])
         .orderBy(col("resale_price").desc_nulls_last()
        )
    )


    # ------------------------------------------------------------
    # 4. ASSIGN ROW NUMBER
    # ------------------------------------------------------------

    df_with_rownum = (q5_input_df.withColumn("row_num",row_number().over(w)))


    # ------------------------------------------------------------
    # 5. VALID DATASET
    # ------------------------------------------------------------

    valid_dataset_df = (df_with_rownum.filter(col("row_num") == 1).drop("row_num"))


    # ------------------------------------------------------------
    # 6. DUPLICATE DATASET
    # ------------------------------------------------------------

    duplicate_failed_df = (df_with_rownum.filter(col("row_num") > 1).drop("row_num"))


    # ------------------------------------------------------------
    # 7. ADD FAILURE REASON TO DUPLICATES
    # ------------------------------------------------------------
    #
    # Keep the same structure as the Q3 failed dataset.
    # ------------------------------------------------------------

    duplicate_failed_df = duplicate_failed_df.withColumn("failed_rule",F.lit("Duplicate record"))


    # ------------------------------------------------------------
    # 8. ADD MISSING VALIDATION COLUMNS IF NECESSARY
    # ------------------------------------------------------------
    #
    # This ensures duplicate_failed_df has the same schema
    # structure as failed_df.
    # ------------------------------------------------------------

    failed_columns = failed_df.columns

    for c in failed_columns:
        if c not in duplicate_failed_df.columns:
            duplicate_failed_df = duplicate_failed_df.withColumn(
                c,
                F.lit(None).cast(
                    failed_df.schema[c].dataType
                )
            )


    # ------------------------------------------------------------
    # 9. REORDER DUPLICATE DATASET
    # ------------------------------------------------------------

    duplicate_failed_df = duplicate_failed_df.select(failed_df.columns)


    # ------------------------------------------------------------
    # 10. MERGE Q3 FAILED + Q5 DUPLICATES
    # ------------------------------------------------------------
    #
    # Q3 failures:
    #   Invalid date
    #   Invalid town
    #   Invalid flat type
    #   Invalid flat model
    #   Invalid storey
    #
    # Q5 failures:
    #   Duplicate record
    #
    # Both are combined into ONE Failed Dataset.
    # ------------------------------------------------------------

    final_failed_df = (failed_df.unionByName(duplicate_failed_df))

    # ------------------------------------------------------------
    # 11. COUNTS
    # ------------------------------------------------------------

    print("\n============================================================")
    print("Q5 DUPLICATE SUMMARY")
    print("============================================================")

    print("Q3 cleaned input:",q5_input_df.count())
    print("Valid records after duplicate handling:",valid_dataset_df.count())
    print("Duplicate records:",duplicate_failed_df.count())
    print("Q3 failed records:",failed_df.count())
    print("Final failed records:", final_failed_df.count())


    # ------------------------------------------------------------
    # 12. PREVIEW VALID DATA
    # ------------------------------------------------------------

    print("\n============================================================")
    print("VALID DATA SAMPLE")
    print("============================================================")

    valid_dataset_df.show(5,truncate=False)


    # ------------------------------------------------------------
    # 13. PREVIEW DUPLICATES
    # ------------------------------------------------------------

    print("\n============================================================")
    print("DUPLICATE DATA SAMPLE")
    print("============================================================")

    duplicate_failed_df.show(5,truncate=False)


    # ------------------------------------------------------------
    # 14. PREVIEW FINAL FAILED DATASET
    # ------------------------------------------------------------

    print("\n============================================================")
    print("FINAL FAILED DATA SAMPLE")
    print("============================================================")

    final_failed_df.select("failed_rule",*[c for c in final_failed_df.columns
                                              if c != "failed_rule"
                                             ]
                          ).show(10,truncate=False)

    # ------------------------------------------------------------
    # 15. SAVE FINAL VALID DATASET
    # ------------------------------------------------------------

    print("\n============================================================")
    print("SAVING FINAL DATASETS")
    print("============================================================")

    valid_dataset_df.toPandas().to_csv(HDB_Dataset_Cleaned,index=False,encoding="utf-8-sig")


    # ------------------------------------------------------------
    # 16. SAVE ONE FINAL FAILED DATASET
    # ------------------------------------------------------------
    final_failed_df.toPandas().to_csv(HDB_Dataset_Failed,index=False,encoding="utf-8-sig")

    # ------------------------------------------------------------
    # 17. FILE INFORMATION
    # ------------------------------------------------------------

    print("\nFiles saved successfully:")
    print("Final Valid Dataset ->",HDB_Dataset_Cleaned)
    print("Final Failed Dataset ->",HDB_Dataset_Failed)

    valid_file_size = os.path.getsize(HDB_Dataset_Cleaned)
    failed_file_size = os.path.getsize(HDB_Dataset_Failed)

    print(f"\nValid dataset size: {valid_file_size} bytes")
    print(f"Failed dataset size: {failed_file_size} bytes")


    # ------------------------------------------------------------
    # 18. FINAL RESULT
    # ------------------------------------------------------------   

    print("Final Valid Dataset  : valid_dataset_df")
    print("Final Failed Dataset : final_failed_df")

    logger.info(f"Valid Data Rows: {valid_dataset_df.count()}")    
    logger.info(f"Failed Data Rows: {final_failed_df.count()}") 
    logger.info(f"Q5 Duplicate data handling completed successfully.\n")
    
    print("\n============================================================")
    print("Q5 DUPLICATE HANDLING COMPLETED SUCCESSFULLY")
    print("============================================================")

except Exception as e:
    logger.exception("Q5 Duplicate data handling process failed")
    print("Q5 Duplicate data handling process failed")
    print("Error:", e)
    raise

In [ ]:
# ============================================================
# Requirement ID : Q6
# Requirement    : Identify Anomalous Resale Prices
# Input          : valid_dataset_df from Q5
# Output         : final Q6 dataset with "reason"
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 20-Aug-2026
# ============================================================

try:
    logger.info("Q6 Anmoalous detection process started.")
    print("\n============================================================")
    print("Q6: ANOMALOUS RESALE PRICE DETECTION")
    print("============================================================")

    # ------------------------------------------------------------
    # 1. USE VALID DATASET FROM Q5
    # ------------------------------------------------------------

    q6_input_df = valid_dataset_df
    print("Input dataset row count:",q6_input_df.count())

    # ------------------------------------------------------------
    # 2. DEFINE COMPARABLE GROUP
    # ------------------------------------------------------------
    #
    # Properties are considered comparable when they share:
    # - Town
    # - Flat Type
    # - Floor Area
    # ------------------------------------------------------------

    comparable_columns = ["town","flat_type","floor_area_sqm"]
    
    # ------------------------------------------------------------
    # 3. CALCULATE Q1 AND Q3
    # ------------------------------------------------------------

    stats_df = (q6_input_df
                .groupBy(*comparable_columns)
                .agg(F.percentile_approx("resale_price",0.25).alias("Q1"),
                     F.percentile_approx("resale_price",0.75).alias("Q3")
                    )
               )

    # ------------------------------------------------------------
    # 4. CALCULATE IQR BOUNDS
    # ------------------------------------------------------------

    stats_df = (stats_df.withColumn("IQR",col("Q3") - col("Q1"))
                .withColumn("lower_bound",col("Q1") - (1.5 * col("IQR"))
                           )
                .withColumn("upper_bound",col("Q3") + (1.5 * col("IQR"))
                           )
               )

    # ------------------------------------------------------------
    # 5. JOIN STATISTICAL BENCHMARKS
    # ------------------------------------------------------------
    joined_df = q6_input_df.join(stats_df,comparable_columns,"left")

    # ------------------------------------------------------------
    # 6. CLASSIFY RECORDS
    # ------------------------------------------------------------
    #
    # Reason:
    #   Normal
    #   Anomalous resale price
    # ------------------------------------------------------------

    final_dataset_df = joined_df.withColumn(
        "anomalydetection",
        when(
            (
                (col("resale_price") < col("lower_bound"))
                |
                (col("resale_price") > col("upper_bound"))
            ),
            lit("Anomalous resale price")
        )
        .otherwise(
            lit("Normal")
        )
    )


    # ------------------------------------------------------------
    # 7. REMOVE STATISTICAL HELPER COLUMNS
    # ------------------------------------------------------------
    #
    # Keep the final dataset clean.
    # ------------------------------------------------------------
    final_dataset_df = final_dataset_df.drop("Q1","Q3","IQR","lower_bound","upper_bound")

    # ------------------------------------------------------------
    # 8. COUNTS
    # ------------------------------------------------------------

    print("\n============================================================")
    print("Q6 SUMMARY")
    print("============================================================")

    print("Total records:",final_dataset_df.count())

    print("Normal records:",final_dataset_df
          .filter(col("anomalydetection") == "Normal")
          .count()
         )

    print("Anomalous records:",final_dataset_df
          .filter(col("anomalydetection") == "Anomalous resale price")
          .count()
         )


    # ------------------------------------------------------------
    # 9. PREVIEW DATA
    # ------------------------------------------------------------

    print("\n============================================================")
    print("Q6 FINAL DATA SAMPLE")
    print("============================================================")

    final_dataset_df.select("month","town","block","street_name","flat_type",
                            "floor_area_sqm","resale_price","anomalydetection").show(10,truncate=False)


    # ------------------------------------------------------------
    # 10. SAVE FINAL DATASET
    # ------------------------------------------------------------

    print("\n============================================================")
    print("SAVING FINAL DATASET")
    print("============================================================")
    final_dataset_df.toPandas().to_csv(HDB_ResalePrice_Anomaly_Detection,index=False,encoding="utf-8-sig")

    print("\nFile saved successfully:")
    print("Q6 dataset ->",HDB_ResalePrice_Anomaly_Detection)

    final_dataset_df.toPandas().to_csv(HDB_Dataset_Cleaned, index=False, encoding="utf-8-sig")
    print("\nFiles saved successfully:")
    print("Valid dataset ->", HDB_Dataset_Cleaned)

    # ------------------------------------------------------------
    # 11. FILE SIZE
    # ------------------------------------------------------------
    file_size = os.path.getsize(HDB_ResalePrice_Anomaly_Detection)
    print(f"Dataset created successfully. Size: {file_size} bytes")


    # ------------------------------------------------------------
    # 12. FINAL PREVIEW
    # ------------------------------------------------------------

    print("\n===== Q6 FINAL DATASET =====")
    final_dataset_df.show(5,truncate=False)   

    logger.info(f"Valid Data Rows: {final_dataset_df.count()}")    
    logger.info(f"Q6 Anmoalous detection completed successfully.\n")
    
    print("\n============================================================")
    print("Q6 ANMOALOUS DETECTION COMPLETED SUCCESSFULLY")
    print("==============================================================")

except Exception as e:
    logger.exception("Q6 Anmoalous detection failed")
    print("Q6 Anmoalous detection failed")
    print("Error:", e)
    raise

In [ ]:
# ============================================================
# Requirement ID : Q7
# Requirement    : Addtional Data Cleaning & Validation
# Input          : final_dataset_df from Q6
# Output         : cleaned_output_df + final_failed_df
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 21-Aug-2026
# ============================================================

try:
    logger.info("Q7 Addtional data cleaning & validation process started.")
    print("\n============================================================")
    print("Q7: ADDTIONAL DATA CLEANING & VALIDATION")
    print("============================================================")

    # ------------------------------------------------------------
    # 1. INPUT DATA FROM Q6
    # ------------------------------------------------------------

    q7_input_df = final_dataset_df

    # ------------------------------------------------------------
    # 2. STANDARDISE CATEGORICAL COLUMNS
    # ------------------------------------------------------------

    q7_input_df = (
        q7_input_df
        .withColumn("town_clean", upper(trim(col("town"))))
        .withColumn("flat_type_clean", upper(trim(col("flat_type"))))
        .withColumn("flat_model_clean", upper(trim(col("flat_model"))))
        .withColumn("storey_range_clean", upper(trim(col("storey_range"))))
    )

    # ------------------------------------------------------------
    # 3. VALIDATION RULES
    # ------------------------------------------------------------

    # Date: YYYY-MM
    is_valid_date = (
        col("month").isNotNull()
        & col("month").cast("string").rlike(
            r"^\d{4}-(0[1-9]|1[0-2])$"
        )
    )

    # Town: not null / blank
    is_valid_town = (
        col("town_clean").isNotNull()
        & (col("town_clean") != "")
    )

    # Flat Type
    valid_flat_types = [
        "1 ROOM",
        "2 ROOM",
        "3 ROOM",
        "4 ROOM",
        "5 ROOM",
        "EXECUTIVE",
        "MULTI-GEN"
    ]

    is_valid_flat_type = (
        col("flat_type_clean").isNotNull()
        & col("flat_type_clean").isin(valid_flat_types)
    )

    # Flat Model: not null / blank
    is_valid_flat_model = (
        col("flat_model_clean").isNotNull()
        & (col("flat_model_clean") != "")
    )

    # Storey Range: 01 TO 03
    storey_lower = F.regexp_extract(
        col("storey_range_clean"),
        r"^([0-9]{2}) TO ([0-9]{2})$",
        1
    ).cast("int")

    storey_upper = F.regexp_extract(
        col("storey_range_clean"),
        r"^([0-9]{2}) TO ([0-9]{2})$",
        2
    ).cast("int")

    is_valid_storey = (
        col("storey_range_clean").isNotNull()
        & col("storey_range_clean").rlike(
            r"^[0-9]{2} TO [0-9]{2}$"
        )
        & (storey_lower > 0)
        & (storey_upper >= storey_lower)
    )

    # Floor Area: > 30 and <= 200 sqm
    is_valid_area = (
        col("floor_area_sqm").isNotNull()
        & (col("floor_area_sqm") > 30)
        & (col("floor_area_sqm") <= 200)
    )

    # Resale Price: > 50,000
    is_valid_price = (
        col("resale_price").isNotNull()
        & (col("resale_price") > 50000)
    )

    # Remaining Lease
    # Q6 contains remaining_lease as a text field,
    # therefore only check that it is present.
    is_valid_lease = (
        col("remaining_lease").isNotNull()
        & (trim(col("remaining_lease")) != "")
    )

    # ------------------------------------------------------------
    # 4. APPLY VALIDATION FLAGS
    # ------------------------------------------------------------

    validated_df = (
        q7_input_df
        .withColumn("is_valid_date", is_valid_date)
        .withColumn("is_valid_town", is_valid_town)
        .withColumn("is_valid_flat_type", is_valid_flat_type)
        .withColumn("is_valid_flat_model", is_valid_flat_model)
        .withColumn("is_valid_storey", is_valid_storey)
        .withColumn("is_valid_area", is_valid_area)
        .withColumn("is_valid_price", is_valid_price)
        .withColumn("is_valid_lease", is_valid_lease)
    )

    # ------------------------------------------------------------
    # 5. FAILURE REASON
    # ------------------------------------------------------------

    validated_df = validated_df.withColumn(
        "failed_rule",
        F.concat_ws(
            "; ",
            when(~col("is_valid_date"), lit("Invalid date")),
            when(~col("is_valid_town"), lit("Invalid town")),
            when(~col("is_valid_flat_type"), lit("Invalid flat type")),
            when(~col("is_valid_flat_model"), lit("Invalid flat model")),
            when(~col("is_valid_storey"), lit("Invalid storey range")),
            when(~col("is_valid_area"), lit("Invalid floor area")),
            when(~col("is_valid_price"), lit("Invalid resale price")),
            when(~col("is_valid_lease"), lit("Invalid remaining lease"))
        )
    )

    # ------------------------------------------------------------
    # 6. CREATE CLEANED DATASET
    # ------------------------------------------------------------

    valid_condition = (
        col("is_valid_date")
        & col("is_valid_town")
        & col("is_valid_flat_type")
        & col("is_valid_flat_model")
        & col("is_valid_storey")
        & col("is_valid_area")
        & col("is_valid_price")
        & col("is_valid_lease")
    )

    cleaned_output_df = validated_df.filter(valid_condition)

    # ------------------------------------------------------------
    # 7. CREATE Q7 FAILED DATASET
    # ------------------------------------------------------------

    q7_failed_df = validated_df.filter(~valid_condition)

    # ------------------------------------------------------------
    # 8. REMOVE INTERNAL VALIDATION COLUMNS
    # ------------------------------------------------------------

    internal_columns = [
        "town_clean",
        "flat_type_clean",
        "flat_model_clean",
        "storey_range_clean",
        "is_valid_date",
        "is_valid_town",
        "is_valid_flat_type",
        "is_valid_flat_model",
        "is_valid_storey",
        "is_valid_area",
        "is_valid_price",
        "is_valid_lease"
    ]

    cleaned_output_df = cleaned_output_df.drop(*internal_columns)

    q7_failed_df = q7_failed_df.drop(*internal_columns)

    # ------------------------------------------------------------
    # 9. MERGE Q7 FAILURES WITH EXISTING FAILED DATASET
    # ------------------------------------------------------------

    # final_failed_df comes from previous questions.
    # Both datasets must have the same columns.

    final_failed_df = final_failed_df.unionByName(
        q7_failed_df,
        allowMissingColumns=True
    )

    # ------------------------------------------------------------
    # 10. SUMMARY
    # ------------------------------------------------------------

    print("\n============================================================")
    print("Q7 VALIDATION SUMMARY")
    print("============================================================")

    print("Q7 Input records       :", q7_input_df.count())
    print("Q7 Cleaned records     :", cleaned_output_df.count())
    print("Q7 Failed records      :", q7_failed_df.count())
    print("Final Failed records   :", final_failed_df.count())

    # ------------------------------------------------------------
    # 11. SAVE CLEANED DATASET
    # ------------------------------------------------------------

    cleaned_output_df.toPandas().to_csv(HDB_Addtional_Data_Cleaning,index=False,encoding="utf-8-sig")

    # ------------------------------------------------------------
    # 12. SAVE FINAL FAILED DATASET
    # ------------------------------------------------------------

    final_failed_path = HDB_Addtional_Data_Cleaning.replace(".csv", "_Final_Failed.csv")
    final_failed_df.toPandas().to_csv(final_failed_path,index=False, encoding="utf-8-sig")

    print("\n============================================================")
    print("FILES SAVED")
    print("============================================================")

    print("Cleaned dataset     ->", HDB_Addtional_Data_Cleaning)
    print("Final failed dataset ->", final_failed_path)



    # SAVE FINAL TO CLEANED and FAILED FOLDER
    cleaned_output_df.toPandas().to_csv(HDB_Dataset_Cleaned, index=False, encoding="utf-8-sig")
    final_failed_df.toPandas().to_csv(HDB_Dataset_Failed, index=False, encoding="utf-8-sig")
    
    print("\nFiles saved successfully:")
    print("Valid dataset ->", HDB_Dataset_Cleaned)
    print("Invalid dataset ->", HDB_Dataset_Failed)

    # ------------------------------------------------------------
    # 13. PREVIEW
    # ------------------------------------------------------------

    print("\n===== CLEANED DATA SAMPLE =====")
    cleaned_output_df.show(5, truncate=False)

    print("\n===== FINAL FAILED DATA SAMPLE =====")

    final_failed_df.select(
        "month",
        "town",
        "flat_type",
        "flat_model",
        "storey_range",
        "floor_area_sqm",
        "resale_price",
        "remaining_lease",
        "failed_rule"
    ).show(10, truncate=False)

   
    logger.info(f"Valid Data Rows: {cleaned_output_df.count()}")    
    logger.info(f"Failed Data Rows: {final_failed_df.count()}")    
    logger.info(f"Q7 Addtional data cleaning & validation completed successfully.\n")
    
    print("\n============================================================")
    print("Q7 COMPLETED SUCCESSFULLY")
    print("============================================================")

except Exception as e:
    logger.exception("Q7 Addtional data cleaning & validation failed")
    print("Q7 Addtional data cleaning & validation failed")
    print("Error:", e)
    raise

In [ ]:
# ============================================================
# Requirement ID : Q8
# Requirement    : Data Transformation Requirements
# Input          : cleaned_output_df from Q7
# Output         : transformed_dataset_df
# Notebook       : HDB_Resale_ETL_Pipeline
# Author         : ILANGO RAMESH
# Created Date   : 21-Aug-2026
# ============================================================


try:
    
    logger.info("Q8 Transformation/Hashed process started.")
    print("\n============================================================")
    print("Q8: DATA TRANSFORMATION")
    print("============================================================")

    # ------------------------------------------------------------
    # 1. INPUT DATA FROM Q7
    # ------------------------------------------------------------

    q8_transformation_df = cleaned_output_df

    # ------------------------------------------------------------
    # 2. PREPARE BLOCK DIGITS
    # ------------------------------------------------------------
    # Remove all non-numeric characters from block.
    # Pad with zeros if fewer than 3 digits.
    #
    # Example:
    # 19   -> 019
    # 123  -> 123
    # ------------------------------------------------------------

    q8_transformation_df = q8_transformation_df.withColumn(
        "block_digits",
        lpad(
            regexp_replace(col("block"), "[^0-9]", ""),
            3,
            "0"
        )
    )

    # ------------------------------------------------------------
    # 3. CALCULATE AVERAGE RESALE PRICE
    # ------------------------------------------------------------
    # Group by:
    #   month + town + flat_type
    #
    # Example:
    # 2017-01 + ANG MO KIO + 2 ROOM
    # Average price = 230000
    # First two digits = 23
    # ------------------------------------------------------------

    avg_price_df = (
        q8_transformation_df
        .groupBy("month", "town", "flat_type")
        .agg(
            F.avg("resale_price").alias("avg_resale_price")
        )
    )

    # Convert average price to integer and extract first 2 digits
    avg_price_df = avg_price_df.withColumn(
        "price_digits",
        substring(
            F.floor(col("avg_resale_price")).cast("string"),
            1,
            2
        )
    )

    # ------------------------------------------------------------
    # 4. JOIN AVERAGE PRICE BACK TO DATASET
    # ------------------------------------------------------------

    q8_transformation_df = q8_transformation_df.join(avg_price_df,["month", "town", "flat_type"],"left")

    # ------------------------------------------------------------
    # 5. EXTRACT MONTH DIGITS
    # ------------------------------------------------------------
    # Example:
    # 2012-01 -> 01
    # 2016-12 -> 12
    # ------------------------------------------------------------

    q8_transformation_df = q8_transformation_df.withColumn("month_digits",substring(col("month"), 6, 2))

    # ------------------------------------------------------------
    # 6. EXTRACT TOWN INITIAL
    # ------------------------------------------------------------

    q8_transformation_df = q8_transformation_df.withColumn("town_initial",substring(col("town"), 1, 1))

    # ------------------------------------------------------------
    # 7. CREATE RESALE IDENTIFIER
    # ------------------------------------------------------------
    #
    # Format:
    # S + 3 block digits + 2 price digits
    #   + 2 month digits + town initial
    #
    # Example:
    # Block       = 19
    # Avg price   = 230000
    # Month       = 2017-01
    # Town        = ANG MO KIO
    #
    # Resale Identifier = S0192301A
    # ------------------------------------------------------------

    q8_transformation_df = q8_transformation_df.withColumn(
        "Resale_Identifier",
        concat(
            lit("S"),
            col("block_digits"),
            col("price_digits"),
            col("month_digits"),
            col("town_initial")
        )
    )

    # ------------------------------------------------------------
    # 8. REMOVE DUPLICATES
    # ------------------------------------------------------------
    # If duplicate records exist, keep the record
    # with the highest resale price.
    #
    # Exclude generated/transformation columns from the
    # duplicate comparison.
    # ------------------------------------------------------------

    duplicate_keys = [
        c for c in q8_transformation_df.columns
        if c not in [
            "resale_price",
            "avg_resale_price",
            "price_digits",
            "block_digits",
            "month_digits",
            "town_initial",
            "Resale_Identifier"
        ]
    ]

    window_spec = (
        Window
        .partitionBy(*duplicate_keys)
        .orderBy(col("resale_price").desc())
    )

    q8_transformation_df = (
        q8_transformation_df
        .withColumn(
            "row_num",
            F.row_number().over(window_spec)
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    # ------------------------------------------------------------
    # 9. HASH RESALE IDENTIFIER
    # ------------------------------------------------------------
    # SHA-256 is used as an irreversible hashing algorithm.
    # It produces a fixed 256-bit hash while preserving
    # practical uniqueness for this dataset.
    # ------------------------------------------------------------

    transformed_dataset_df = q8_transformation_df.withColumn("Resale_Identifier_Hash",sha2(col("Resale_Identifier"), 256))

    # ------------------------------------------------------------
    # 10. REMOVE INTERNAL TRANSFORMATION COLUMNS
    # ------------------------------------------------------------

    transformed_dataset_df = transformed_dataset_df.drop(
        "block_digits",
        "avg_resale_price",
        "price_digits",
        "month_digits",
        "town_initial"
    )

    # ------------------------------------------------------------
    # 11. SUMMARY
    # ------------------------------------------------------------

    print("\nInput records :", cleaned_output_df.count())
    print("Transformed records :",transformed_dataset_df.count())

    # ------------------------------------------------------------
    # 12. SAMPLE OUTPUT
    # ------------------------------------------------------------

    print("\n===== TRANSFORMED DATA SAMPLE =====")

    transformed_dataset_df.select(
        "month",
        "town",
        "block",
        "flat_type",
        "resale_price",
        "Resale_Identifier",
        "Resale_Identifier_Hash"
    ).show(10, truncate=False)

    # ------------------------------------------------------------
    # 13. SAVE FINAL TRANSFORMED DATASET
    # ------------------------------------------------------------

    transformed_dataset_df.toPandas().to_csv(HDB_Transformed_Hashed, index=False, encoding="utf-8-sig")

    # SAVE FINAL TO CLEANED and FAILED FOLDER
    transformed_dataset_df.toPandas().to_csv(HDB_Dataset_Cleaned, index=False, encoding="utf-8-sig")  
    
    print("\nFiles saved successfully:")
    print("Valid dataset ->", HDB_Dataset_Cleaned)
   
    print("Transformation/Hashed dataset ->",HDB_Transformed_Hashed)
    file_size = os.path.getsize(HDB_Transformed_Hashed)

    print(f"Dataset created successfully. Size: {file_size} bytes")

    logger.info(f"Valid Data Rows: {transformed_dataset_df.count()}")       
    logger.info(f"Q8 Transformation/Hashed completed successfully.\n")
    
    print("\n============================================================")
    print("Q8 TRANSFORMED PROCESS COMPLETED SUCCESSFULLY")
    print("============================================================")

except Exception as e:
    logger.exception("Q8 Transformation/Hashed process failed")
    print("Q8 Transformation/Hashed process failed")
    print("Error:", e)
    raise
